# SeatGuard — Station 1: Leather Anomaly Detection (Anomalib PatchCore)

Trains PatchCore on the **MVTec AD leather category** (normal images only).  
PatchCore is an anomaly detection model — it only needs clean/normal images to train.  
It learns what normal leather looks like, then flags anything different.

**Runtime:** GPU → Runtime → Change runtime type → T4 GPU

**Dataset:** MVTec AD — leather category (~245 normal training images)  
**License:** CC BY-NC-SA 4.0 (research/development use — fine for training)  
**Expected output:** `patchcore.onnx` (~300 MB) for `models/station_1/`

## Step 1 — Install Anomalib and dependencies

In [ ]:
# Anomalib v1.x — pinned for stability
!pip install anomalib==1.1.0 --quiet
!pip install onnx onnxruntime --quiet

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

import anomalib
print(f"Anomalib: {anomalib.__version__}")

## Step 2 — Download MVTec AD leather category

MVTec hosts the dataset directly. The leather category is ~400 MB.  
No API key needed — direct download.

In [ ]:
import os

!wget -q --show-progress \
    'https://www.mydrive.ch/shares/38523/dd19ccefcf8d2f9dc3fd719e68743b95/download/420938113-1629952094/leather.tar.xz' \
    -O leather.tar.xz

!mkdir -p /content/mvtec
!tar -xf leather.tar.xz -C /content/mvtec
!rm leather.tar.xz

# Verify structure
for root, dirs, files in os.walk("/content/mvtec/leather"):
    level = root.replace("/content/mvtec/leather", "").count(os.sep)
    if level <= 2:
        indent = "  " * level
        n_files = len(files)
        print(f"{indent}{os.path.basename(root)}/ ({n_files} files)")

## Step 3 — Verify dataset

MVTec leather structure should be:
```
leather/
  train/
    good/        ← ~245 normal images for training
  test/
    good/        ← normal test images
    cut/         ← defect test images
    fold/
    glue/
    pinholes/
    scratch/
  ground_truth/  ← pixel masks
```

In [ ]:
import os
from pathlib import Path

leather_root = Path("/content/mvtec/leather")

train_good = leather_root / "train" / "good"
test_good  = leather_root / "test"  / "good"

train_imgs = list(train_good.glob("*.png")) + list(train_good.glob("*.jpg"))
test_imgs  = list(test_good.glob("*.png"))  + list(test_good.glob("*.jpg"))

print(f"Train normal images : {len(train_imgs)}")
print(f"Test  normal images : {len(test_imgs)}")

# List defect categories in test
test_categories = [d.name for d in (leather_root / "test").iterdir() if d.is_dir()]
print(f"Test categories     : {test_categories}")

if len(train_imgs) < 100:
    print("WARNING: fewer training images than expected — check download")
else:
    print("\nDataset looks good!")

## Step 4 — Train PatchCore

PatchCore training on ~245 images takes **3–5 minutes** on T4 (it's not gradient-based — it builds a memory bank of normal patch features).

Key settings:
- `input_size`: 256×256 — matches the SeatGuard ONNX engine's expected input
- `coreset_sampling_ratio`: 0.1 — keeps 10% of patches (memory bank size vs speed tradeoff)
- `num_neighbors`: 9 — patch neighbourhood for anomaly scoring

In [ ]:
from anomalib.data import MVTec
from anomalib.models import Patchcore
from anomalib.engine import Engine
from anomalib.callbacks import ModelCheckpoint

# Data module — MVTec leather
datamodule = MVTec(
    root="/content/mvtec",
    category="leather",
    image_size=(256, 256),
    train_batch_size=32,
    eval_batch_size=32,
    num_workers=2,
)

# PatchCore model
model = Patchcore(
    backbone="wide_resnet50_2",
    layers_list=["layer2", "layer3"],
    pre_trained=True,
    coreset_sampling_ratio=0.1,
    num_neighbors=9,
)

# Engine (Anomalib v1.x trainer wrapper)
engine = Engine(
    accelerator="gpu",
    devices=1,
    default_root_dir="seatguard_station1",
)

# Train — PatchCore has no epochs (single-pass memory bank construction)
engine.fit(model=model, datamodule=datamodule)
print("\nPatchCore training complete.")

## Step 5 — Evaluate on test set

Reports AUROC (Area Under ROC curve) — the standard metric for anomaly detection.  
Target: AUROC > 0.90

In [ ]:
# Evaluate
results = engine.test(model=model, datamodule=datamodule)
print("\nTest results:")
for k, v in results[0].items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

## Step 6 — Export to ONNX

Exports the PatchCore model to ONNX format compatible with SeatGuard's `OnnxPatchCoreEngine`.

The engine expects:
- Input: `"input"` float32 `[2, 3, 256, 256]` — batch of 2 (duplicate single image)
- Outputs: `pred_score`, `pred_label`, `anomaly_map`, `pred_mask`

In [ ]:
import torch
import os
from pathlib import Path

# Export to ONNX via Anomalib engine
engine.export(
    model=model,
    export_type="onnx",
    export_root="seatguard_station1/exports",
    input_size=(256, 256),
)

# Find the exported file
import glob
onnx_files = glob.glob("seatguard_station1/**/*.onnx", recursive=True)
print("ONNX files found:")
for f in onnx_files:
    size_mb = os.path.getsize(f) / 1e6
    print(f"  {f}  ({size_mb:.1f} MB)")

onnx_path = onnx_files[0] if onnx_files else None
print(f"\nUsing: {onnx_path}")

## Step 7 — Verify ONNX output names match SeatGuard engine

SeatGuard's `OnnxPatchCoreEngine` expects specific output names.  
This cell checks and renames them if needed.

In [ ]:
import onnx

m = onnx.load(onnx_path)
onnx.checker.check_model(m)
print("ONNX check passed.")

print("\nInputs:")
for inp in m.graph.input:
    shape = [d.dim_value for d in inp.type.tensor_type.shape.dim]
    print(f"  '{inp.name}': {shape}")

print("\nOutputs:")
for out in m.graph.output:
    shape = [d.dim_value for d in out.type.tensor_type.shape.dim]
    print(f"  '{out.name}': {shape}")

# Expected by SeatGuard OnnxPatchCoreEngine:
EXPECTED_INPUT  = "input"
EXPECTED_OUTPUTS = ["pred_score", "pred_label", "anomaly_map", "pred_mask"]

actual_inputs  = [i.name for i in m.graph.input]
actual_outputs = [o.name for o in m.graph.output]

print("\n--- Compatibility check ---")
print(f"Input  '{EXPECTED_INPUT}' present: {EXPECTED_INPUT in actual_inputs}")
for o in EXPECTED_OUTPUTS:
    print(f"Output '{o}' present: {o in actual_outputs}")

## Step 8 — Rename outputs if needed

If the compatibility check above showed any outputs missing, run this cell to rename them.  
If all outputs are present, skip this cell.

In [ ]:
import onnx
from onnx import helper

m = onnx.load(onnx_path)
actual_outputs = [o.name for o in m.graph.output]
actual_inputs  = [i.name for i in m.graph.input]

print("Actual inputs :", actual_inputs)
print("Actual outputs:", actual_outputs)

# Build rename map: actual_name → seatguard_name
# Anomalib v1 typically outputs: pred_score, pred_label, anomaly_map, pred_mask
# but may use different names — adjust the LEFT side if needed
OUTPUT_RENAME = {
    # actual name from Anomalib → SeatGuard expected name
    # Add mappings only if names differ
    # e.g. "score": "pred_score"
}

INPUT_RENAME = {
    # e.g. "input.1": "input"
}

def rename_io(model, input_map, output_map):
    """Rename ONNX graph inputs and outputs in-place."""
    # Rename in graph outputs
    for output in model.graph.output:
        if output.name in output_map:
            output.name = output_map[output.name]
    # Rename in last nodes
    for node in model.graph.node:
        node.output[:] = [output_map.get(o, o) for o in node.output]
        node.input[:]  = [input_map.get(i, i)  for i in node.input]
    # Rename graph inputs
    for inp in model.graph.input:
        if inp.name in input_map:
            inp.name = input_map[inp.name]
    return model

if OUTPUT_RENAME or INPUT_RENAME:
    m = rename_io(m, INPUT_RENAME, OUTPUT_RENAME)
    onnx.save(m, onnx_path)
    print("\nRenamed and saved.")
    # Verify
    m2 = onnx.load(onnx_path)
    print("New outputs:", [o.name for o in m2.graph.output])
else:
    print("No renaming needed — outputs already match SeatGuard expectations.")

## Step 9 — Smoke test inference

In [ ]:
import onnxruntime as ort
import numpy as np
import cv2
from pathlib import Path

sess = ort.InferenceSession(onnx_path, providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
print("Providers:", sess.get_providers())

input_name  = sess.get_inputs()[0].name
output_names = [o.name for o in sess.get_outputs()]
print(f"Input : '{input_name}'")
print(f"Outputs: {output_names}")

# Test on a real normal image
test_img_path = list(Path("/content/mvtec/leather/test/good").glob("*.png"))[0]
img = cv2.imread(str(test_img_path))
rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
resized = cv2.resize(rgb, (256, 256)).astype(np.float32) / 255.0
chw = resized.transpose(2, 0, 1)
batch = np.stack([chw, chw], axis=0)   # batch of 2

outputs = sess.run(None, {input_name: batch})
pred_score = float(outputs[0][0])
pred_label = bool(outputs[1][0])
print(f"\nNormal image inference:")
print(f"  pred_score : {pred_score:.4f}  (should be LOW for normal image)")
print(f"  pred_label : {pred_label}  (should be False for normal image)")

# Test on a defect image
defect_img_path = list(Path("/content/mvtec/leather/test/cut").glob("*.png"))[0]
img2 = cv2.imread(str(defect_img_path))
rgb2 = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)
resized2 = cv2.resize(rgb2, (256, 256)).astype(np.float32) / 255.0
chw2 = resized2.transpose(2, 0, 1)
batch2 = np.stack([chw2, chw2], axis=0)

outputs2 = sess.run(None, {input_name: batch2})
pred_score2 = float(outputs2[0][0])
pred_label2 = bool(outputs2[1][0])
print(f"\nDefect image (cut) inference:")
print(f"  pred_score : {pred_score2:.4f}  (should be HIGHER than normal)")
print(f"  pred_label : {pred_label2}  (should be True for defect)")

## Step 10 — Download the ONNX file

Place it at `models/station_1/patchcore.onnx` in the SeatGuard project.

In [ ]:
import shutil, os
from google.colab import files

output_name = "patchcore.onnx"
shutil.copy(onnx_path, output_name)

size_mb = os.path.getsize(output_name) / 1e6
print(f"File size: {size_mb:.1f} MB")

if size_mb > 400:
    print("NOTE: File is large. Download may take a moment.")

files.download(output_name)
print("\nDownload started.")
print("Place this file at:  models/station_1/patchcore.onnx")

## Notes

### After downloading
1. Copy `patchcore.onnx` to `models/station_1/patchcore.onnx`
2. Update `configs/stations/station_1_leather.yaml`:
```yaml
model:
  backend: "onnx"
  model_type: "anomalib_patchcore"
  weights_path: "models/station_1/patchcore.onnx"
  version: "v1.0.0"
  threshold: 0.45
  pass_below: 0.40
  fail_above: 0.60
```
3. Bring the model back to chat for integration

### Expected results
| Metric | Acceptable | Good |
|--------|-----------|------|
| AUROC  | > 0.85    | > 0.95 |
| F1 at threshold | > 0.70 | > 0.85 |

PatchCore on MVTec leather typically achieves AUROC ~0.99 — one of the best anomaly detection results on this dataset.

### Why no epochs?
PatchCore does not use gradient-based training. It:
1. Passes all normal training images through a pretrained ResNet backbone
2. Extracts patch features
3. Builds a coreset (compressed memory bank) from those features
4. At inference, compares new patches against the memory bank

This is why training takes only 3–5 minutes even on CPU.

### If the download URL fails (Step 2)
The MVTec direct download URL sometimes changes. Alternative:
```python
# Use Anomalib's built-in dataset downloader
from anomalib.data import MVTec
datamodule = MVTec(root="/content/mvtec", category="leather")
datamodule.prepare_data()   # downloads automatically
```